# Somo la 13 - Kumbukumbu ya Wakala


## Setup

Notebook hii inaonyesha jinsi ya kuunda wakala wa uhifadhi wa usafiri kwa kutumia **kumbukumbu endelevu** kwa kutumia **Microsoft Agent Framework** (MAF).

Utajifunza jinsi aina tofauti za kumbukumbu za wakala — za kufanya kazi, za muda mfupi, na za muda mrefu — zinavyoathiri jinsi wakala anavyohifadhi na kutumia taarifa katika mazungumzo mbalimbali.

**Mahitaji ya awali:**
- Mradi wa Azure AI Foundry wenye modeli ya mazungumzo iliyowekwa (km. `gpt-4o-mini`).
- Umeingia kwenye Azure CLI — tumia `az login` kwenye terminal yako.
- `AZURE_AI_PROJECT_ENDPOINT` — kiungo cha mradi wako wa Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jina la modeli yako iliyowekwa.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Aina za Kumbukumbu za Wakala

Wakala wa AI wanaweza kutumia aina tofauti za kumbukumbu, kila moja ikiwa na matumizi yake ya kipekee:

### Kumbukumbu ya Kufanya Kazi
Mstari wa mazungumzo yenyewe — ujumbe unaobadilishana katika kikao kimoja. Wakala anaweza kurejelea ujumbe wa awali katika mstari huo huo ili kudumisha muunganiko. Katika MAF hii huundwa kupitia **`agent.create_session()`**, ambayo hurejesha `AgentSession`.

### Kumbukumbu ya Muda Mfupi
Taarifa zinazodumu kwa muda wa kazi au kikao lakini hazihifadhiwi kwa kudumu. Kwa mfano, wakala anaweza kukusanya ukweli wakati wa mazungumzo ya kupanga ya mizunguko mingi na kuyatumia kutoa ratiba ya mwisho.

### Kumbukumbu ya Muda Mrefu
Mapendeleo na ukweli unaodumu **katika vipindi vya vikao**. Mtumiaji anayerudiha haapaswi kurudia vikwazo vya chakula au mtindo wa kusafiri. Kumbukumbu ya muda mrefu kawaida huungwa mkono na hazina ya nje — hifadhidata, faili, au index ya vector — na huoneshwa kwa wakala kupitia zana.


## Kumbukumbu Inayofanya Kazi na Vikao

Aina rahisi zaidi ya kumbukumbu ni kikao cha mazungumzo. Unapopita kikao hicho hicho (kilichotengenezwa kupitia `agent.create_session()`) kwa miito inayoendelea ya `agent.run()`, wakala anaona historia kamili ya mazungumzo hayo na anaweza kukumbuka maelezo ya awali.

Tuanze kuunda wakala wa usafiri na kuonyesha kumbukumbu inayofanya kazi.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Wakala alikumbuka bajeti kwa usahihi kwa sababu ujumbe wote wawili unashiriki kikao kimoja. Hii ni **kumbukumbu ya kazi** — ipo tu kwa maisha ya kikao hicho.

### Nini hutokea na mfululizo mpya?

Ikiwa tunaunda kikao **kipya**, wakala hana kumbukumbu ya mazungumzo ya awali:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Mfumo wa Kumbukumbu za Muda Mrefu

Ili kukumbuka mapendeleo ya mtumiaji **katika vikao mbalimbali**, tunahitaji hifadhi ya kudumu inayokaa nje ya mwelekeo wa mazungumzo. Mwakilishi anapata hifadhi hii kupitia **zana** — kazi anazoweza kuita kuhifadhi na kupata taarifa.

Hapo chini tunatekeleza hifadhi rahisi ya mapendeleo ndani ya kumbukumbu (katika utengenezaji ungeiweka nyuma na hifadhidata au kiashiria cha vector) na kuifichua kama zana ambazo mwakilishi anaweza kutumia.

### Miundo
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Hali ya 1 — Mtumiaji wa mara ya kwanza anahifadhi safari ya maadhimisho

Sarah anatembelea kwa mara ya kwanza. Wakala anapaswa kuhifadhi mapendeleo yake kupitia zana na kuyatumia kupendekeza hoteli.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Hali ya 2 — Sarah anarudi wiki kadhaa baadaye

Sarah huanzisha **mashina mpya kabisa** (kuiga kikao kipya). Kumbukumbu ya kazi ni tupu, lakini hifadhi ya mapendeleo ya muda mrefu bado ina taarifa zake. Wakala anapaswa kuizipata na kuzitumia kubinafsisha mapendekezo.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Muhtasari

Katika somo hili ulijifunza aina tatu za kumbukumbu za wakala na jinsi ya kuzitekeleza kwa kutumia Microsoft Agent Framework:

| Aina ya Kumbukumbu | Mekanismo ya MAF | Muda wa Kuishi |
|---|---|---|
| **Kazi** | `agent.create_session()` | Mazungumzo moja |
| **Muda mfupi** | Muktadha uliokusanywa ndani ya thread | Kazi / kikao kimoja |
| **Muda mrefu** | Hifadhi ya nje inayopatikana kupitia kazi za `@tool` | Kikao mbali mbali |

### Vidokezo Muhimu
1. **`agent.create_session()`** hutoa kumbukumbu ya kazi — wakala anaona historia yote ya mazungumzo ndani ya kikao.
2. **Vikao vipya hupoteza muktadha** — bila kumbukumbu za muda mrefu wakala hawezi kukumbuka mazungumzo ya zamani.
3. **Kazi za `@tool`** huunganisha pengo — zinamruhusu wakala kuhifadhi na kupata habari kutoka kwenye hifadhi ya kudumu.
4. **Ubinafsishaji huboresha kwa muda** — kadri mapendeleo yanavyohifadhiwa, ndivyo mapendekezo ya wakala yanavyokuwa bora.

### Matumizi halisi ya Dunia
- **Huduma kwa Wateja**: Kumbuka historia na mapendeleo ya mteja
- **Msaidizi Binafsi**: Dumu muktadha kwa siku au wiki
- **Huduma za Afya**: Fuatilia taarifa na mapendeleo ya mgonjwa
- **Biashara Mtandaoni**: Manunuzi binafsi kulingana na historia

### Hatua Zifuatazo
- Badilisha dict ya kumbukumbu ya ndani na database au hifadhi ya vector (mfano Azure AI Search)
- Ongeza muda wa kumalizika kwa kumbukumbu za habari nyeti kwa muda
- Tengeneza mifumo ya mawakala wengi yenye kumbukumbu ya pamoja
- Chunguza daftari la Cognee kwa kumbukumbu zinazoendeshwa na grafu ya maarifa


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kiarifu**:  
Hati hii imetafsiriwa kwa kutumia huduma ya kutafsiri kwa AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi usahihi, tafadhali fahamu kuwa tafsiri za mashine zinaweza kuwa na makosa au upotofu. Hati asilia katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo thabiti. Kwa taarifa muhimu, tafsiri ya kitaalamu kwa binadamu inashauriwa. Hatubebei wajibu wowote kwa kutoelewana au tafsiri potofu zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
